# Hybrid Quantum Neural Network (HQNN) on top of pretrained ResNet18

In [1]:
from pathlib import Path
from typing import Optional

import kagglehub
import numpy as np
import pennylane as qml
import torch
import torch.nn as nn
from PIL import Image
import os
import re
import random
from torch.utils.data import DataLoader, Dataset, Subset, TensorDataset, random_split
from torchvision import transforms
from tqdm import tqdm

torch.manual_seed(0)
random.seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


## Dataset 

In [2]:
class VMMRDB_Dataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.class_to_idx = {}

        unique_classes = set()
        for class_name in sorted(os.listdir(root_dir)):
            class_path = os.path.join(root_dir, class_name)
            if os.path.isdir(class_path):
                stripped = re.sub(r"_\d{4}$", "", class_name)
                unique_classes.add(stripped)

        for idx, class_name in enumerate(sorted(unique_classes)):
            self.class_to_idx[class_name] = idx

        for class_name in sorted(os.listdir(root_dir)):
            class_path = os.path.join(root_dir, class_name)
            if os.path.isdir(class_path):
                stripped = re.sub(r"_\d{4}$", "", class_name)
                idx = self.class_to_idx[stripped]
                for img_name in os.listdir(class_path):
                    if img_name.lower().endswith((".png", ".jpg", ".jpeg")):
                        self.image_paths.append(os.path.join(class_path, img_name))
                        self.labels.append(idx)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

In [3]:
path = kagglehub.dataset_download("prabashwara/vmmrdb-dataset")
print("Path to dataset files:", path)

# Speed: cap samples after the 80/20 split (None = use all). Delete feature .pt caches after changing.
TRAIN_SAMPLE_CAP = None
TEST_SAMPLE_CAP = None

batch_size = 64

train_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(
            brightness=0.2,
            contrast=0.2,
            saturation=0.2,
            hue=0.1,
        ),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize(
            [0.4367, 0.4331, 0.4279], [0.2685, 0.2652, 0.2675]
        ),
    ]
)

test_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            [0.4367, 0.4331, 0.4279], [0.2685, 0.2652, 0.2675]
        ),
    ]
)

dataset_full = VMMRDB_Dataset(path, transform=train_transform)
num_classes = len(dataset_full.class_to_idx)
print("num_classes:", num_classes)

train_size = int(0.8 * len(dataset_full))
test_size = len(dataset_full) - train_size
generator = torch.Generator().manual_seed(0)
dataset_train, dataset_test = random_split(
    dataset_full, [train_size, test_size], generator=generator
)
dataset_test.dataset.transform = test_transform

if TRAIN_SAMPLE_CAP is not None:
    n = min(TRAIN_SAMPLE_CAP, len(dataset_train))
    dataset_train = Subset(dataset_train, range(n))
if TEST_SAMPLE_CAP is not None:
    n = min(TEST_SAMPLE_CAP, len(dataset_test))
    dataset_test = Subset(dataset_test, range(n))

# num_workers=0: on Windows + Jupyter, worker processes often deadlock (tqdm stuck at 0%, idle CPU).
train_loader = DataLoader(
    dataset_train,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)
test_loader = DataLoader(
    dataset_test,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)

Path to dataset files: C:\Users\godca\.cache\kagglehub\datasets\prabashwara\vmmrdb-dataset\versions\1
num_classes: 1175


## ResNet18

In [4]:
class BasicBlock(nn.Module):
    def __init__(self, in_channel, out_channel, stride=1):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(
            in_channel, out_channel, kernel_size=3, padding=1, stride=stride, bias=False
        )
        self.batch_norm1 = nn.BatchNorm2d(out_channel)
        self.conv2 = nn.Conv2d(
            out_channel, out_channel, kernel_size=3, stride=1, padding=1, bias=False
        )
        self.batch_norm2 = nn.BatchNorm2d(out_channel)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channel != out_channel:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channel, out_channel, kernel_size=1, stride=stride, bias=False
                ),
                nn.BatchNorm2d(out_channel),
            )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        out = self.relu(self.batch_norm1(self.conv1(x)))
        out = self.batch_norm2(self.conv2(out))
        out += self.shortcut(x)
        out = self.relu(out)
        return out


class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=1000):
        super(ResNet, self).__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.layer1 = self._make_layer(block, 64, layers[0], stride=1)
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, block, out_channels, blocks, stride):
        layers = [block(self.in_channels, out_channels, stride)]
        self.in_channels = out_channels
        for _ in range(1, blocks):
            layers.append(block(self.in_channels, out_channels))
        return nn.Sequential(*layers)

    def forward_features(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        return torch.flatten(x, 1)

    def forward(self, x):
        return self.fc(self.forward_features(x))


def ResNet18(num_classes):
    return ResNet(BasicBlock, [2, 2, 2, 2], num_classes)

## Hybrid head + load `test_resnet18.pth`

Replace the final `fc` with: `Linear(512 → n_qubits)` → angle embedding → `StronglyEntanglingLayers` → Pauli-Z expectations → `Linear(n_qubits → num_classes)`.

In [5]:
def make_quantum_layer(n_qubits: int, n_layers: int):
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, interface="torch", diff_method="backprop")
    def circuit(inputs, weights):
        qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="X")
        qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
        return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

    weight_shapes = {"weights": (n_layers, n_qubits, 3)}
    return qml.qnn.TorchLayer(circuit, weight_shapes)


class HybridHead(nn.Module):
    """512-D ResNet embedding -> VQC -> logits."""

    def __init__(self, in_dim: int, num_classes: int, n_qubits: int, n_layers: int):
        super().__init__()
        self.pre_q = nn.Linear(in_dim, n_qubits)
        self.scale = nn.Parameter(torch.ones(n_qubits))
        self.qlayer = make_quantum_layer(n_qubits, n_layers)
        self.head = nn.Linear(n_qubits, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        angles = torch.tanh(self.pre_q(x)) * self.scale * np.pi
        q_out = self.qlayer(angles)
        return self.head(q_out)


def find_pretrained_path() -> Path:
    candidates = [
        Path("test_resnet18.pth"),
        Path("VMMRdb/test_resnet18.pth"),
    ]
    for p in candidates:
        if p.is_file():
            return p.resolve()
    raise FileNotFoundError(
        "test_resnet18.pth not found. Place it in VMMRdb/ or repo root (cwd when you run the notebook)."
    )


FEATURE_DIM = 512
n_qubits = 6
n_vqc_layers = 2

ckpt_path = find_pretrained_path()
print("Loading:", ckpt_path)
try:
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
except TypeError:
    checkpoint = torch.load(ckpt_path, map_location=device)
if "model_state" not in checkpoint:
    raise KeyError("Expected checkpoint key 'model_state' (from vmmrdb_cnn).")

model = ResNet18(num_classes=num_classes).to(device)
model.load_state_dict(checkpoint["model_state"])
model.fc = HybridHead(FEATURE_DIM, num_classes, n_qubits, n_vqc_layers).to(device)

for name, p in model.named_parameters():
    if not name.startswith("fc."):
        p.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("trainable parameters:", trainable)

Loading: C:\Users\godca\Downloads\Last sem\Hybrid-Quantum-Neural-Networks-for-Image-Classification\VMMRdb\test_resnet18.pth
trainable parameters: 11345


## Train HQNN head for 20 epochs

In [ ]:
USE_FEATURE_CACHE = True
PRECOMPUTE_BATCH = 64
HEAD_BATCH_SIZE = 64
CACHE_DIR = Path("VMMRdb") if Path("VMMRdb").is_dir() else Path(".")
train_cache = CACHE_DIR / "hqnn_cached_train.pt"
test_cache = CACHE_DIR / "hqnn_cached_test.pt"


def _torch_load_cpu(path: Path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def materialize_feature_caches():
    model.eval()
    pm = getattr(train_loader, "pin_memory", False)
    for base_loader, path, tag in [
        (train_loader, train_cache, "train"),
        (test_loader, test_cache, "test"),
    ]:
        if path.exists():
            print("Using existing cache:", path)
            continue
        feats_chunks, y_chunks = [], []
        fast_loader = DataLoader(
            base_loader.dataset,
            batch_size=PRECOMPUTE_BATCH,
            shuffle=False,
            num_workers=0,
            pin_memory=pm,
        )
        n_batches = len(fast_loader)
        print(
            f"Caching {path.name}: {n_batches} batches (first batch loads images from disk; wait or see note below)",
            flush=True,
        )
        with torch.no_grad():
            for bi, (x, y) in enumerate(
                tqdm(fast_loader, desc=f"ResNet features -> {path.name}")
            ):
                if bi == 0:
                    print("  first batch in, shape:", tuple(x.shape), flush=True)
                x = x.to(device, non_blocking=True)
                if device.type == "cuda":
                    with torch.autocast(device_type="cuda", dtype=torch.float16):
                        z = model.forward_features(x)
                else:
                    z = model.forward_features(x)
                feats_chunks.append(z.float().cpu())
                y_chunks.append(y)
        blob = {
            "feats": torch.cat(feats_chunks, dim=0),
            "labels": torch.cat(y_chunks, dim=0),
        }
        torch.save(blob, path)
        print("Wrote", path, blob["feats"].shape)


if USE_FEATURE_CACHE:
    materialize_feature_caches()
    print("Loading feature tensors into RAM (big files can sit here a few minutes, low CPU)...", flush=True)
    tr = _torch_load_cpu(train_cache)
    te = _torch_load_cpu(test_cache)
    print("Train feats:", tr["feats"].shape, "Test feats:", te["feats"].shape, flush=True)
    train_loop_loader = DataLoader(
        TensorDataset(tr["feats"], tr["labels"]),
        batch_size=HEAD_BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=(device.type == "cuda"),
    )
    test_loop_loader = DataLoader(
        TensorDataset(te["feats"], te["labels"]),
        batch_size=HEAD_BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=(device.type == "cuda"),
    )
    active_net = model.fc
else:
    train_loop_loader, test_loop_loader = train_loader, test_loader
    active_net = model


def run_epoch(
    net: nn.Module,
    loader,
    opt: Optional[torch.optim.Optimizer],
    loss_fn,
):
    train = opt is not None
    net.train(train)
    total_loss = 0.0
    correct = 0
    n = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    it = tqdm(loader, leave=True) if train else loader
    print(
        f"run_epoch ({'train' if train else 'eval'}): {len(loader)} batches",
        flush=True,
    )
    with ctx:
        for bi, (x, y) in enumerate(it):
            if bi == 0:
                print("  first batch:", tuple(x.shape), flush=True)
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            if train:
                opt.zero_grad()
            logits = net(x)
            loss = loss_fn(logits, y)
            if train:
                loss.backward()
                opt.step()
            total_loss += loss.item() * x.size(0)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            n += x.size(0)
            if train:
                it.set_postfix(loss=loss.item())
    return total_loss / n, correct / n


epochs = 20
lr = 1e-3
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, active_net.parameters()), lr=lr
)

for epoch in range(1, epochs + 1):
    tr_loss, tr_acc = run_epoch(active_net, train_loop_loader, optimizer, loss_fn)
    te_loss, te_acc = run_epoch(active_net, test_loop_loader, None, loss_fn)
    print(
        f"epoch {epoch:02d}/{epochs}  train loss {tr_loss:.4f} acc {tr_acc:.4f}  "
        f"test loss {te_loss:.4f} acc {te_acc:.4f}"
    )

Caching hqnn_cached_train.pt: 3564 batches (first batch loads images from disk; wait or see note below)


ResNet features -> hqnn_cached_train.pt:   0%|          | 0/3564 [00:00<?, ?it/s]

  first batch in, shape: (64, 3, 224, 224)


ResNet features -> hqnn_cached_train.pt: 100%|██████████| 3564/3564 [09:50<00:00,  6.04it/s]


Wrote hqnn_cached_train.pt torch.Size([228068, 512])
Caching hqnn_cached_test.pt: 891 batches (first batch loads images from disk; wait or see note below)


...

  0%|          | 0/3564 [00:00<?, ?it/s]

run_epoch (train): 3564 batches
  first batch: (64, 512)


100%|██████████| 3564/3564 [02:33<00:00, 23.17it/s, loss=4.19]

run_epoch (eval): 891 batches
  first batch: (64, 512)


epoch 20/20  train loss 4.1892 acc 0.1384  test loss 4.2811 acc 0.1349


## save hybrid checkpoint


In [7]:
out = Path("vmmr_hqnn_head.pth")
torch.save({"model_state": model.state_dict(), "num_classes": num_classes, "n_qubits": n_qubits}, out)
print("Saved", out)

Saved vmmr_hqnn_head.pth
